In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('../src'))
if module_path not in sys.path:
    sys.path.append(module_path)
#from vsopy import util, data
import astropy.units as u
import numpy as np

import vsopy.util
import vsopy.data

OBJ_NAME='SZ Lyn'
SESSION_TAG='2026/20260123'
IMAGE_ROOT = '/srv/public/img'
WORK_ROOT = '/srv/public'

session = vsopy.util.Session(tag=SESSION_TAG, name=OBJ_NAME)
layout = vsopy.util.WorkLayout(WORK_ROOT)
session_layout = layout.get_session(session)
IMAGE_DIR = vsopy.util.ImageLayout(IMAGE_ROOT).get_images(session).lights_dir / 'V'
sd = vsopy.data.StarData(layout.charts.root_dir)
viewer = None

Select image, plate solve and calibrate

In [ ]:
import ccdproc as ccdp
import vsopy.reduce

ifc = ccdp.ImageFileCollection(IMAGE_DIR)
path = IMAGE_DIR / ifc.summary['file'][30]
image  = vsopy.reduce.load_and_solve(path, session_layout.solved_dir)
matcher = vsopy.reduce.CalibrationMatcher(layout.calibr_dir, temp_tolerance=2*u.K)
cal = matcher.match(image.header)
reduced = vsopy.reduce.calibrate_image(image,
                        dark=cal.dark,
                        flat=cal.flat)


Calculate image stats

In [ ]:
from astropy.stats import sigma_clipped_stats
mean, median, std = sigma_clipped_stats(reduced.data, sigma=3.0)
mean, median, std

Detect sources with IRAFStarFinder.  set initial FWHM to make average
sharpness close to 1

In [ ]:
from photutils.detection import IRAFStarFinder
from astropy.nddata import CCDData

image = CCDData.subtract(reduced, median * reduced.unit)
finder = IRAFStarFinder(fwhm=5.0, threshold=5.0*std * reduced.unit)
sources = finder.find_stars(image.data * image.unit)
#np.mean(sources['sharpness']), np.std(sources['sharpness'])
sources


In [ ]:
from photutils.psf import fit_2dgaussian

fit_2dgaussian(image.data * image.unit,
            xypos=list(zip(sources['xcentroid'], sources['ycentroid'])),
            fwhm=4.5,
            fix_fwhm=False,
            #error=reduced.uncertainty,
            fit_shape=15).results

Investigate how initial FWHM affect fit results

In [ ]:
from photutils.psf import fit_2dgaussian
from astropy.table import QTable

fwhm_init = np.linspace(.5, 8, 16)
fit_res = [fit_2dgaussian(image.data * image.unit,
            xypos=list(zip(sources['xcentroid'], sources['ycentroid'])),
            fwhm=fwhm,
            fix_fwhm=False,
            #error=reduced.uncertainty,
            fit_shape=15).results
              for fwhm in fwhm_init]

def filtered_stats(table):
    mn = np.mean(table['fwhm_fit'])
    filtered = table[(table['fwhm_fit'] > 0.5*mn) & (table['fwhm_fit'] < 2*mn)]
    return {'count': len(filtered),
            'init':np.mean(table['fwhm_init']),
            'fit': np.mean(filtered['fwhm_fit']),
            'err': np.std(filtered['fwhm_fit']),
            'unc': np.mean(filtered['fwhm_err'])}


table = QTable([filtered_stats(res) for res in fit_res])
table

Apparently fitted FWHM is stable over wide range of initial condition 
and converges to 4.5 pixels

In [ ]:
from plotnine import (ggplot, aes, geom_point, geom_errorbar,
                      scale_y_continuous)

(ggplot(table)
+ aes(x='init', y='fit')
+ geom_point()
+ geom_errorbar(aes(ymin='fit-err', ymax='fit+err'))
+ geom_errorbar(aes(ymin='fit-unc', ymax='fit+unc'), color='red')
+ scale_y_continuous(limits=(2, None)))

Investigate how fit shape affects the results

In [ ]:

fit_shape = range(3, 21, 2)
fit_res = [fit_2dgaussian(image.data * image.unit,
            xypos=list(zip(sources['xcentroid'], sources['ycentroid'])),
            fwhm=4.5,
            fix_fwhm=False,
            fit_shape=shape).results
              for shape in fit_shape]


table1 = QTable([filtered_stats(res) for res in fit_res])
table1['shape'] = fit_shape
table1

Looking at the FWHM uncertainty the sweet spot for the shape is around 13,
which corresonds to FWHM*3

In [ ]:
(ggplot(table1)
+ aes(x='shape', y='fit')
+ geom_point()
+ geom_errorbar(aes(ymin='fit-err', ymax='fit+err'))
+ geom_errorbar(aes(ymin='fit-unc', ymax='fit+unc'), color='red')
+ scale_y_continuous(limits=(0, None)))

Finally, try elliptic gaussian

In [ ]:
from photutils.psf import GaussianPRF, PSFPhotometry
from photutils.utils import CutoutImage
from photutils.background import LocalBackground

model = GaussianPRF()
model.theta.fixed = False
model.theta.bounds = (-90.0, 90.0)
model.x_fwhm.fixed = False
model.y_fwhm.fixed = False

fit_shape=13
photometry = PSFPhotometry(model, fit_shape=fit_shape,
                           aperture_radius=fit_shape//3,
                           localbkg_estimator=LocalBackground(fit_shape*1.5, fit_shape*2))

xypos=np.atleast_2d(list(zip(sources['xcentroid'], sources['ycentroid'])))
flux_init = []
for yxpos in xypos[:, ::-1]:
    cutout = CutoutImage(reduced, yxpos, (fit_shape, fit_shape))
    flux_init.append(np.sum(cutout.data))

flux_init <<= reduced.unit

init_params = QTable()
init_params['x'] = xypos[:, 0]
init_params['y'] = xypos[:, 1]
init_params['theta'] = [0] * len(xypos)
#init_params['flux'] = flux_init

photometry(reduced, init_params=init_params, error=reduced.uncertainty)
photometry.fit_params

In [ ]:
def filtered_fit_params(table):
    fwhm_fit = (table['x_fwhm_fit'] + table['y_fwhm_fit']) / 2
    mn = np.mean(fwhm_fit)
    return table[(fwhm_fit > 0.5*mn) & (fwhm_fit < 2*mn)]

table2 = filtered_fit_params(photometry.fit_params)
table2


In [ ]:
from plotnine import (scale_x_log10)
(ggplot(table2)
+ aes(x='flux_fit', y='flux_err')
+ geom_point()
+scale_x_log10())